## Imports & Setup

In [0]:
import json
import time
from pyspark.sql.functions import current_timestamp,row_number,to_timestamp,lit, col, max as spark_max
from delta.tables import DeltaTable
from pyspark.sql.window import Window

status = "SUCCESS"
error_message = None
records_read = 0
records_written = 0

## Widgets

In [0]:
import json
from pyspark.sql.functions import current_timestamp

# INPUTS

run_id = dbutils.widgets.get("run_id")

table_metadata = json.loads(dbutils.widgets.get("table_metadata"))
table_parameters = json.loads(dbutils.widgets.get("table_parameters"))

# EXTRACT VARIABLES (GENERIC)

table_id = int(table_metadata["table_id"])
table_name = table_metadata["table_name"]

bronze_schema = table_metadata["bronze_schema"]
silver_schema = table_metadata["silver_schema"]

load_type = table_parameters.get("load_type")
primary_key = table_parameters.get("primary_key")
watermark_column = table_parameters.get("watermark_column")

# DYNAMIC TABLES

bronze_table = f"banking.{bronze_schema}.{table_name}"
silver_table = f"banking.{silver_schema}.{table_name}"

print(f"Processing table: {table_name}")
print(f"Bronze: {bronze_table}")
print(f"Silver: {silver_table}")
print(f"Load Type: {load_type}")

## Silver Layer Processing & Incremental Load (Full / Append / Merge with Watermarking)

In [0]:
try:
    bronze_df = spark.table(bronze_table)

    print("BRONZE RAW COUNT:", bronze_df.count())

    last_watermark = None

    if load_type in ["APPEND", "MERGE"] and watermark_column:

        watermark_df = spark.sql(f"""
            SELECT last_watermark_value
            FROM banking.metadata.table_watermarks
            WHERE table_id = {table_id}
        """)

        if watermark_df.take(1):
            last_watermark = watermark_df.first()["last_watermark_value"]

        print("LAST WATERMARK =", last_watermark)

        print("COUNT BEFORE FILTER =", bronze_df.count())

        if last_watermark is not None:
            bronze_df = bronze_df.filter(
                col(watermark_column) > last_watermark
            )

        print("COUNT AFTER FILTER =", bronze_df.count())

    records_read = bronze_df.count()

    print("Records to process:", records_read)

    # Add audit columns

    bronze_df = (
        bronze_df
        .withColumn("insert_timestamp", current_timestamp())
        .withColumn("update_timestamp", current_timestamp())
    )

    # Create Silver table

    spark.sql("create schema if not exists banking.silver")

    if not spark.catalog.tableExists(silver_table):
        (
            bronze_df
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
        )
        records_written = records_read

    else:
        # FULL
        if load_type == "FULL":

            (
                bronze_df
                .write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(silver_table)
            )
            records_written = records_read

        # APPEND

        elif load_type == "APPEND":

            (
                bronze_df
                .write
                .format("delta")
                .mode("append")
                .saveAsTable(silver_table)
            )
            records_written = records_read

        # MERGE
      
        elif load_type == "MERGE":

            if not primary_key:
                raise ValueError("Primary key required for MERGE.")
             # REMOVE DUPLICATES BEFORE MERGE
            window = Window.partitionBy(primary_key).orderBy(
                col(watermark_column).desc()
            )

            bronze_df = bronze_df.withColumn(
                "rn", row_number().over(window)
            ).filter(col("rn") == 1).drop("rn")

            delta_table = DeltaTable.forName(spark, silver_table)

            merge_condition = f"t.{primary_key} = s.{primary_key}"

            (
                delta_table.alias("t")
                .merge(
                    bronze_df.alias("s"),
                    merge_condition
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )

            records_written = records_read

        else:
            raise ValueError("Unsupported load_type")

    # Update Watermark (APPEND & MERGE)
   
    if load_type in ["APPEND", "MERGE"] and watermark_column:

        max_value = bronze_df.agg(
            spark_max(col(watermark_column))
        ).collect()[0][0]

        if max_value:

            spark.sql(f"""
                MERGE INTO banking.metadata.table_watermarks t
                USING (SELECT {table_id} AS table_id) s
                ON t.table_id = s.table_id
                WHEN MATCHED THEN UPDATE SET
                    last_watermark_value = '{max_value}',
                    last_updated_at = current_timestamp()
                WHEN NOT MATCHED THEN
                    INSERT (table_id, last_watermark_value, last_updated_at)
                    VALUES ({table_id}, '{max_value}', current_timestamp())
            """)

    print("Silver Load Completed Successfully.")


except Exception as e:

    status = "FAILED"
    error_message = str(e)
    print("Error Occurred:", error_message)
    raise


finally:

    end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

    # Insert into Audit Table
  
    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET
            end_time = TIMESTAMP('{end_time}'),
            status = '{status}',
            number_of_records = {records_read},
            error_message = {'NULL' if not error_message else "'" + error_message.replace("'", "") + "'"}
        WHERE table_id = {table_id} and run_id = {run_id}
    """)

    print("Audit record inserted.")